In [1]:
!pip install librosa pandas numpy tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!unzip "/content/drive/Shareddrives/CS 163 Project/VocalSet.zip" -d "/content/"

Archive:  /content/drive/Shareddrives/CS 163 Project/VocalSet.zip
   creating: /content/FULL/
  inflating: /content/FULL/.DS_Store  
   creating: /content/__MACOSX/
   creating: /content/__MACOSX/FULL/
  inflating: /content/__MACOSX/FULL/._.DS_Store  
   creating: /content/FULL/female9/
  inflating: /content/FULL/female9/.DS_Store  
   creating: /content/__MACOSX/FULL/female9/
  inflating: /content/__MACOSX/FULL/female9/._.DS_Store  
   creating: /content/FULL/female9/excerpts/
   creating: /content/FULL/female9/excerpts/vibrato/
  inflating: /content/FULL/female9/excerpts/vibrato/f9_caro_vibrato.wav  
  inflating: /content/FULL/female9/excerpts/vibrato/f9_row_vibrato.wav  
  inflating: /content/FULL/female9/excerpts/vibrato/f9_dona_vibrato.wav  
   creating: /content/FULL/female9/excerpts/straight/
  inflating: /content/FULL/female9/excerpts/straight/f9_dona_straight.wav  
  inflating: /content/FULL/female9/excerpts/straight/f9_row_straight.wav  
  inflating: /content/FULL/female9/exc

In [ ]:
import glob

# Collect all .wav audio file paths inside the VocalSet dataset directory
files = glob.glob("/content/FULL/**/*.wav", recursive=True)

print("Total wav files:", len(files))
print(files[:10])

Total wav files: 3613
['/content/FULL/male6/scales/vibrato/m6_scales_vibrato_e.wav', '/content/FULL/male6/scales/vibrato/m6_scales_vibrato_o.wav', '/content/FULL/male6/scales/vibrato/m6_scales_vibrato_u.wav', '/content/FULL/male6/scales/vibrato/m6_scales_vibrato_a.wav', '/content/FULL/male6/scales/vibrato/m6_scales_vibrato_i.wav', '/content/FULL/male6/scales/breathy/m6_scales_breathy_u.wav', '/content/FULL/male6/scales/breathy/m6_scales_breathy_a.wav', '/content/FULL/male6/scales/breathy/m6_scales_breathy_i.wav', '/content/FULL/male6/scales/breathy/m6_scales_breathy_e.wav', '/content/FULL/male6/scales/breathy/m6_scales_breathy_o.wav']


In [ ]:
# Filter the file list to ensure only .wav audio files are included
# (this helps avoid hidden files like .DS_Store that may appear in the dataset)
files = [f for f in files if f.lower().endswith(".wav")]
import pandas as pd
# Extract the vocal technique label from the directory structure
# The technique is the folder name before the file (e.g., belt, breathy, vibrato)
tech = pd.Series([f.split("/")[-2] for f in files], name="technique")
tech.value_counts()

,count
technique,
slow_piano,397
slow_forte,395
fast_forte,394
fast_piano,386
straight,361
vibrato,255
belt,205
lip_trill,202
breathy,200


belt -> high energy
breathy -> low energy

In [ ]:
# Create an empty list to store only the audio files we want to keep
filtered_files = []
for f in files:
     # Extract the vocal technique from the folder name
    # The technique is the directory before the filename
    technique = f.split("/")[-2]

    # Keep only the ones with the vocal technique we want
    if technique == "belt" or technique == "breathy":
        filtered_files.append(f)

# Replace the original file list with the filtered subset
files = filtered_files

# Create a pandas Series of the remaining techniques
# This helps confirm that only belt and breathy remain in the dataset
tech_filtered = pd.Series([f.split("/")[-2] for f in files], name="technique")
display(tech_filtered.value_counts())

,count
technique,
belt,205
breathy,200


In [ ]:
!pip -q install librosa tqdm praat-parselmouth

import numpy as np
import librosa
from tqdm import tqdm
import parselmouth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 80.4 MB/s eta 0:00:00


In [ ]:
# Function to estimate the fundamental frequency (pitch) of an audio signal
# using the YIN algorithm provided by librosa
def compute_f0_yin(y, sr, fmin=50, fmax=500):

    # Compute pitch values across the audio signal
    # fmin and fmax define the expected pitch range for human vocals
    f0 = librosa.yin(y, fmin=fmin, fmax=fmax, sr=sr)

    # Remove any invalid pitch values (NaN or infinite values)
    f0 = f0[np.isfinite(f0)]

    # Return the average pitch across the audio clip
    # If no valid values exist, return NaN
    return float(np.mean(f0)) if len(f0) else np.nan

# Function to compute the Harmonic-to-Noise Ratio (HNR) using Praat via parselmouth
# HNR measures how harmonic (clean) versus noisy a voice signal is
def compute_hnr_praat(file_path, min_pitch=50):

    # Load the audio file using parselmouth (Praat interface)
    snd = parselmouth.Sound(file_path)

    # Compute harmonicity values across the signal
    harm = snd.to_harmonicity_cc(time_step=0.01, minimum_pitch=min_pitch)

    # Extract the harmonicity values
    vals = harm.values[0]

    # Remove invalid values
    vals = vals[np.isfinite(vals)]

    # Return the mean HNR value for the audio clip
    return float(np.mean(vals)) if len(vals) else np.nan


# Function to extract all acoustic features from a single audio file
def extract_features(file_path):

    # Load the audio file using librosa
    # sr=None keeps the original sampling rate of the file
    # mono=True converts stereo audio into a single channel
    y, sr = librosa.load(file_path, sr=None, mono=True)

    # Compute the fundamental frequency (pitch)
    f0 = compute_f0_yin(y, sr)

    # Compute the spectral centroid
    # This represents the "brightness" of the sound
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))

    # Compute the spectral rolloff
    # This indicates the frequency below which most spectral energy lies
    rolloff = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))

    # Compute the harmonic-to-noise ratio (voice clarity vs noise)
    # This uses Praat through parselmouth
    hnr = compute_hnr_praat(file_path)

    # Return all extracted features
    return f0, centroid, rolloff, hnr

In [ ]:
# Initialize an empty list to store the extracted feature data
rows = []

# Counter to track files that fail during feature extraction
failed = 0

# Loop through every audio file in the filtered dataset
# tqdm displays a progress bar so we can track extraction progress
for i, fp in enumerate(tqdm(files)):
    try:
        # Extract the vocal technique from the folder name in the file path
        # This will be either "belt" or "breathy"
        technique = fp.split("/")[-2]

        # Map the technique to the corresponding class label
        # belt → high energy vocals
        # breathy → low energy vocals
        if technique == "belt":
            label = "high_energy"
        else:
            label = "low_energy"

        # Extract acoustic features from the audio file
        # Returns pitch (f0), spectral centroid, spectral rolloff, and HNR
        f0, centroid, rolloff, hnr = extract_features(fp)

        # Store the extracted features and metadata in a dictionary
        rows.append({
            "sample_id": i,              # primary key for each sample
            "file_path": fp,             # original file path kept for traceability
            "technique": technique,
            "class": label,
            "f0": f0,
            "spectral_centroid": centroid,
            "spectral_rolloff": rolloff,
            "hnr": hnr
        })

    # If an error occurs during feature extraction (e.g., corrupted audio file),
    # increment the failure counter and skip that file
    except Exception:
        failed += 1

# Convert the collected rows into a pandas DataFrame for analysis
df = pd.DataFrame(rows)

# Display how many files were successfully processed
print("Extracted rows:", df.shape[0])

# Display how many files failed during feature extraction
print("Failed files:", failed)

# Show the first few rows of the dataset to verify the extracted features
df.head()

  0%|          | 1/405 [00:03<25:38,  3.81s/it]


KeyboardInterrupt: 

In [ ]:
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# Basic validity filters (safe defaults)
df = df[(df["f0"] > 50) & (df["f0"] < 500)]
df = df[(df["spectral_centroid"] > 0) & (df["spectral_rolloff"] > 0)]

print(df.shape)
print(df["class"].value_counts())
print(df.groupby("class").mean(numeric_only=True))
df.describe()

(405, 7)
class
high_energy    205
low_energy     200
Name: count, dtype: int64
                     f0  spectral_centroid  spectral_rolloff       hnr
class                                                                 
high_energy  245.917628        3124.064844       5398.233422 -5.926238
low_energy   252.360847        3193.747154       6319.907019 -8.311923


,f0,spectral_centroid,spectral_rolloff,hnr
count,405.000000,405.000000,405.000000,405.000000
mean,249.099464,3158.475861,5853.380877,-7.104354
std,71.337303,788.739949,2289.375966,14.711259
min,158.326740,1420.762795,1634.101230,-86.201807
25%,177.893369,2565.909322,4430.523476,-13.111713
50%,208.100950,3044.766479,5352.753679,-3.952880
75%,318.655896,3590.667159,6649.531191,2.084921
max,367.749826,6613.677655,17127.241109,23.766282


In [ ]:
out_path = "/content/drive/Shareddrives/CS 163 Project/vocal_features.csv"
df.to_csv(out_path, index=False)
print("Saved to:", out_path)

Saved to: /content/drive/MyDrive/163 Project/vocal_features.csv


---
## EDA — Exploratory Data Analysis
**Scope:** Descriptive Statistics + Visualizations  
Uses `df` from the feature extraction above (405 samples, belt / breathy).

In [ ]:
# Load completed feature CSV — run this cell to skip re-running feature extraction
import pandas as pd
import numpy as np

CSV_PATH = '/content/drive/Shareddrives/CS 163 Project/vocal_features.csv'
df = pd.read_csv(CSV_PATH)

print('Loaded:', df.shape)
print(df['technique'].value_counts())
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DRIVE_PATH = '/content/drive/Shareddrives/CS 163 Project/'
COLORS = {'belt': '#2196F3', 'breathy': '#FF9800'}
FEATURES = ['f0', 'spectral_centroid', 'spectral_rolloff', 'hnr']
LABELS = {
    'f0': 'f0 (Hz)',
    'spectral_centroid': 'Spectral Centroid (Hz)',
    'spectral_rolloff': 'Spectral Rolloff (Hz)',
    'hnr': 'HNR (dB)'
}
print('EDA setup complete.')

In [ ]:
# Data quality check
print('=== MISSING VALUES ===')
print(df[FEATURES].isnull().sum())

print('\n=== DUPLICATE ROWS ===')
print(f'Duplicates: {df.duplicated().sum()}')

print('\n=== HNR RANGE ===')
print(f'Min: {df["hnr"].min():.2f} dB | Max: {df["hnr"].max():.2f} dB')

hnr_outliers = df[df['hnr'] < -20]
print(f'\nHNR outliers (< -20 dB): {len(hnr_outliers)} rows')
print(hnr_outliers['technique'].value_counts())

In [ ]:
# Filter HNR artifacts before computing stats
df_clean = df[df['hnr'] >= -20].copy()
removed = len(df) - len(df_clean)
print(f'Rows removed: {removed} ({removed/len(df)*100:.1f}%)')
print(f'Clean dataset: {df_clean.shape}')
print(df_clean['technique'].value_counts())

In [ ]:
# Overall descriptive statistics
print('=== OVERALL DESCRIPTIVE STATS ===')
print(df_clean[FEATURES].describe().round(3).to_string())

In [ ]:
# Per-class descriptive statistics (belt vs. breathy)
print('=== PER-CLASS STATS ===\n')
for technique in ['belt', 'breathy']:
    print(f'--- {technique.upper()} ---')
    subset = df_clean[df_clean['technique'] == technique][FEATURES]
    print(subset.describe().round(3).to_string())
    print()

In [ ]:
# Correlation matrix
corr = df_clean[FEATURES].corr().round(3)
print('=== CORRELATION MATRIX ===')
print(corr.to_string())

In [ ]:
# Histograms — feature distributions overlaid by class
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Feature Distributions by Vocal Technique', fontsize=14, fontweight='bold')

for ax, feature in zip(axes.flatten(), FEATURES):
    for technique, color in COLORS.items():
        vals = df_clean[df_clean['technique'] == technique][feature]
        ax.hist(vals, bins=30, alpha=0.6, color=color, label=technique,
                edgecolor='white', linewidth=0.5)
    ax.set_title(LABELS[feature])
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.savefig(DRIVE_PATH + 'eda_hist_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_hist_features.png')

In [ ]:
# Boxplots — feature spread by class
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Feature Spread by Vocal Technique (Boxplots)', fontsize=14, fontweight='bold')

for ax, feature in zip(axes.flatten(), FEATURES):
    sns.boxplot(data=df_clean, x='technique', y=feature, ax=ax,
                palette=COLORS, order=['belt', 'breathy'],
                width=0.5, flierprops=dict(marker='o', markersize=3, alpha=0.5))
    ax.set_title(LABELS[feature])
    ax.set_xlabel('')
    ax.set_ylabel('Value')

plt.tight_layout()
plt.savefig(DRIVE_PATH + 'eda_boxplot_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_boxplot_features.png')

In [ ]:
# Scatter plot — f0 vs spectral centroid colored by class
fig, ax = plt.subplots(figsize=(8, 6))

for technique, color in COLORS.items():
    subset = df_clean[df_clean['technique'] == technique]
    ax.scatter(subset['f0'], subset['spectral_centroid'],
               alpha=0.55, color=color, label=technique, s=45, edgecolors='none')

ax.set_xlabel('Fundamental Frequency f0 (Hz)', fontsize=11)
ax.set_ylabel('Spectral Centroid (Hz)', fontsize=11)
ax.set_title('f0 vs Spectral Centroid by Vocal Technique', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(DRIVE_PATH + 'eda_scatter_f0_centroid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_scatter_f0_centroid.png')

In [ ]:
# Correlation heatmap
corr = df_clean[FEATURES].corr()
tick_labels = ['f0', 'Centroid', 'Rolloff', 'HNR']

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax,
            xticklabels=tick_labels, yticklabels=tick_labels,
            square=True, linewidths=0.5)
ax.set_title('Acoustic Feature Correlation Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(DRIVE_PATH + 'eda_heatmap_corr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_heatmap_corr.png')

## Preliminary Insights

| Hypothesis | Feature(s) | What to look for |
|------------|-----------|------------------|
| Belt has higher pitch | f0 | Belt median f0 > Breathy median f0 |
| Belt has brighter timbre | spectral_centroid, spectral_rolloff | Belt values higher |
| Belt is more periodic | hnr | Belt HNR > Breathy HNR |
| Centroid and rolloff correlate | both | Strong positive correlation expected |

**HNR note:** Values below -20 dB were filtered as extraction artifacts.  
If removal was class-skewed, note it as a data quality limitation in the report.